In [28]:
pip install langchain

Note: you may need to restart the kernel to use updated packages.


In [29]:
%pip install -U pypdf

Note: you may need to restart the kernel to use updated packages.


In [30]:
%pip install -U langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [32]:
from langchain_community.document_loaders import PyPDFLoader
loader=PyPDFLoader(r"C:\Users\chakr\Downloads\Onepiece.pdf")
data=loader.load()
print(len(data))

3


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter=RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," "],
    chunk_size=500,
    chunk_overlap=0
)
chunk=text_splitter.split_documents(data)
print(len(chunk))

14


In [34]:
from secret_key import api_key
from langchain_openai import OpenAIEmbeddings
embeddings=OpenAIEmbeddings(
    api_key=api_key,
    model="text-embedding-3-small"
)

In [35]:
from langchain_community.vectorstores import FAISS
vectorstore=FAISS.from_documents(
    chunk,
    embeddings
)

In [36]:
retriever=vectorstore.as_retriever(
    search_kwargs={"k":3}
)

In [37]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(
    api_key=api_key,
    temperature=0.7
)

In [38]:
from langchain_core.prompts import ChatPromptTemplate
prompt=ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Context:
{context}

Question:
{question}

If the answer is not available in the context,
say "I don't know based on the provided articles."
""")

In [39]:
%pip install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib

Note: you may need to restart the kernel to use updated packages.


In [40]:
import os
import pickle
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build
SCOPES=[
    "https://www.googleapis.com/auth/gmail.modify",
    "https://www.googleapis.com/auth/gmail.send"
]
creds=None
if os.path.exists("token.json"):
    with open("token.json","rb")as token:
        creds=pickle.load(token)
if not creds or not creds.valid:
    if creds and creds.expired and creds.refresh_token:
        creds.refresh(Request())
    else:
        flow=InstalledAppFlow.from_client_secrets_file(
            r"C:\Users\chakr\Downloads\client_secret_255703380212-l7unbrgt4m9gg89k1l8lk7qug8j215ht.apps.googleusercontent.com.json",
            SCOPES
        )
        creds=flow.run_local_server(port=0)
    with open("token.json","wb") as token:
        pickle.dump(creds,token)
service=build(
    "gmail",
    "v1",
    credentials=creds
)
print("Gmail connected successfully")

Gmail connected successfully


In [41]:
results = service.users().messages().list(
    userId="me",
    labelIds=["INBOX"],
    q="is:unread"
).execute()

messages = results.get("messages", [])

print("Unread emails:", len(messages))

Unread emails: 100


In [42]:
import base64

def get_email_body(payload):
    
    body = ""

    # Simple email
    if "body" in payload:
        data = payload["body"].get("data")

        if data:
            body = base64.urlsafe_b64decode(
                data
            ).decode(
                "utf-8",
                errors="ignore"
            )

    # Multipart email
    if "parts" in payload:

        for part in payload["parts"]:

            if part.get("mimeType") == "text/plain":

                data = part.get("body", {}).get("data")

                if data:
                    body = base64.urlsafe_b64decode(
                        data
                    ).decode(
                        "utf-8",
                        errors="ignore"
                    )

                    return body

    return body

In [43]:
if not messages:

    print("No unread emails found.")

else:

    message_id = messages[0]["id"]

    email_data = service.users().messages().get(
        userId="me",
        id=message_id,
        format="full"
    ).execute()

    payload = email_data["payload"]

    headers = payload.get("headers", [])

    sender = ""
    subject = ""

    for header in headers:

        if header["name"].lower() == "from":
            sender = header["value"]

        if header["name"].lower() == "subject":
            subject = header["value"]

    question = get_email_body(payload)

    print("Sender:")
    print(sender)

    print("\nSubject:")
    print(subject)

    print("\nCustomer Question:")
    print(question)

Sender:
chakri reddy <chakrireddy9291@gmail.com>

Subject:
Requesting to answer

Customer Question:
who is luffy?



In [44]:

docs=retriever.invoke(question)
context="\n\n".join(
    doc.page_content for doc in docs
)
print("Retrieved Context:")
print(context)

Retrieved Context:
Monkey D. Luffy is the captain of the Straw Hat Pirates. He is energetic, optimistic, loyal, and
determined. His greatest goal is to find the legendary treasure known as the One Piece and become the
Pirate King. Luffy is famous for his unusual fighting style, strong will, and ability to inspire people around
him. His journey is not only about becoming powerful; it is also about friendship, freedom, and protecting
people who are important to him.
The World of One Piece

The Straw Hat Pirates and Their Journey
The Straw Hat Crew
The Straw Hat Pirates are the central group of characters in the story. Each member has a different
dream, personality, and skill. Luffy is the captain; Roronoa Zoro is a swordsman who aims to become
the world's greatest swordsman; Nami is a talented navigator who dreams of creating a map of the
world; Usopp is a sharpshooter and storyteller who wants to become a brave warrior of the sea.

One Piece: An Introduction to the Legendary
 Anime
Over

In [45]:
message = prompt.invoke({
    "question": question,
    "context": context
})

answer = llm.invoke(message)

print("AI Answer:")
print(answer.content)

AI Answer:
Luffy is the captain of the Straw Hat Pirates.


In [46]:
import re

match = re.search(
    r'[\w\.-]+@[\w\.-]+\.\w+',
    sender
)

if match:
    customer_email = match.group(0)
    print("Customer email:", customer_email)
else:
    print("Could not find customer email.")

Customer email: chakrireddy9291@gmail.com


In [47]:
from email.mime.text import MIMEText
import base64

reply = MIMEText(answer.content)

reply["To"] = customer_email
reply["Subject"] = "Re: " + subject

raw_message = base64.urlsafe_b64encode(
    reply.as_bytes()
).decode("utf-8")

body = {
    "raw": raw_message
}

sent_message = service.users().messages().send(
    userId="me",
    body=body
).execute()

print("Reply sent successfully!")

Reply sent successfully!
